# AlexNet Kernel Restriction Study — Phase 2 (FP32 & QAT INT8)

**Author:** Rafael  
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)  
**Goal:** Train and compare all Phase 2 kernel-restriction variants in FP32, then apply
Quantization-Aware Training (QAT) to obtain INT8 versions. Report **Top-1** and **Top-5**
accuracy for every model.

| Model | Kernel strategy | Notes |
|---|---|---|
| AlexNetTV | 11×11 → 5×5 → 3×3 (original) | Pretrained torchvision AlexNet; reference baseline |
| AlexNet3x3 | All 3×3 | From-scratch; same AlexNet head |
| AlexNet2x2 | All 2×2 | From-scratch; smallest even kernel; no Winograd |
| AlexNetStacked | Stacked 3×3 pairs + BN | Conv-BN-ReLU triples; recovers 5×5 receptive field |
| AlexNetMixed | Alternating 3×3 and 2×2 | Heterogeneous kernel restriction |
| AlexNetSmallKernel | All 3×3, GlobalAvgPool | Lightweight: 1.6M params, single-linear head |

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps)
5. FP32 training
6. QAT training
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table
10. Persist experiment summary

## 1. Configuration & reproducibility

Everything that controls the experiment lives in a single `ExperimentConfig`
dict. The seed is fixed and `set_seed` propagates it to `random`, `numpy`,
and `torch` (CPU & CUDA).


In [ ]:
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, load_best_model, convert_to_int8,
    build_comparison_table, create_results_summary, disk_mb,
)
from configs.loader import load_config

from models import (
    build_alexnet,
    build_alexnet_3x3, AlexNet3x3,
    build_alexnet_2x2, AlexNet2x2,
    build_alexnet_stacked, AlexNetStacked,
    build_alexnet_mixed, AlexNetMixed,
    build_alexnet_smallkernel, AlexNetSmallKernel,
)

torch.backends.quantized.engine = "fbgemm"

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "alexnet_phase2"
RESULTS_DIR = project_root / "results" / "alexnet_phase2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

## 2. Dataset & loaders

We use Tiny-ImageNet-200 from Kaggle. The train folder is split 90/10 into
train/val with a **seeded generator** so the split is identical on every run.

* Training transforms: light augmentation (random crop + hflip + color jitter)
* Validation transforms: deterministic resize + center crop

Note that we build two `ImageFolder` objects (one per transform) and index
them with the same `Subset` indices — this keeps train-time augmentation
separate from val-time deterministic preprocessing.


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train


In [ ]:
train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

## 3. Model shape check

Six Phase 2 variants — all from `models/alexnet_variants.py`:

| Name | Kernel sizes | BN? | Head |
|---|---|---|---|
| AlexNetTV | 11×11, 5×5, 3×3 (original) | No | 3-layer MLP |
| AlexNet3x3 | All 3×3 | No | 3-layer MLP |
| AlexNet2x2 | All 2×2 | No | Linear(256, 200) |
| AlexNetStacked | Stacked 3×3 pairs | **Yes** | 3-layer MLP |
| AlexNetMixed | Alternating 3×3/2×2 | No | Linear(256, 200) |
| AlexNetSmallKernel | All 3×3, GAP | No | Linear(256, 200) |

In [ ]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetTV",          build_alexnet),
    ("AlexNet3x3",         build_alexnet_3x3),
    ("AlexNet2x2",         build_alexnet_2x2),
    ("AlexNetStacked",     build_alexnet_stacked),
    ("AlexNetMixed",       build_alexnet_mixed),
    ("AlexNetSmallKernel", build_alexnet_smallkernel),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    print(f"{label:20s} OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

## 4. Model registration

Fuse maps are index-based paths inside `features` for flat-Sequential models.
`AlexNetStacked` uses Conv-BN-ReLU triples; all others use Conv-ReLU pairs.
All six models support QAT — no skip set needed.

In [ ]:
# Conv-ReLU pairs (no BN) — shared by AlexNetTV, 3x3, 2x2, Mixed, SmallKernel
FUSE_CONV_RELU = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]

# Conv-BN-ReLU triples — AlexNetStacked (10 conv layers, 5 stages × 2 blocks)
FUSE_MAP_STACKED = [
    ["0","1","2"],["3","4","5"],      # Stage 1 (MaxPool at 6)
    ["7","8","9"],["10","11","12"],   # Stage 2 (MaxPool at 13)
    ["14","15","16"],["17","18","19"],# Stage 3
    ["20","21","22"],["23","24","25"],# Stage 4
    ["26","27","28"],["29","30","31"],# Stage 5
]

MODEL_REGISTRY.clear()
register_model("alexnet_tv",           build_alexnet,            fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_3x3",          build_alexnet_3x3,        fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_2x2",          build_alexnet_2x2,        fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_stacked",      build_alexnet_stacked,    fuse_map=FUSE_MAP_STACKED, fuse_root_attr="features", lr=1e-3)
register_model("alexnet_mixed",        build_alexnet_mixed,      fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_small_kernel", build_alexnet_smallkernel,fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)

print("Registered:", list(MODEL_REGISTRY))

## 5. FP32 training

In [ ]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-phase2",
        group="fp32-phase2",
        name=f"{name}_fp32",
        config={**asdict(model_cfg), "arch": name, "phase": "fp32"},
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
    )
    results = trainer.fit()
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

## 6. Quantization-Aware Training (QAT)

All six models support QAT — fuse maps are defined for every architecture.
QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [ ]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-phase2",
        group="qat-phase2",
        name=f"{name}_qat",
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
        },
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
    )
    results = trainer.fit()
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.

In [11]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {}
for name in MODEL_REGISTRY:
    int8_models[name] = convert_to_int8(qat_models[name].cpu().eval())

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    m = m.cpu().eval()
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(m, val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                      num_classes=data_cfg.num_classes)
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

INT8 conversion done.
alexnet_fp32           | loss=3.5021 | top1=22.08% | top5=47.30%
alexnet_3x3            | loss=4.4869 | top1=7.52% | top5=22.41%
alexnet_small_kernel   | loss=4.2778 | top1=10.77% | top5=28.73%


## 8. FP32 evaluation — reload best checkpoints

In [ ]:
CTORS = {
    "alexnet_tv":           build_alexnet,
    "alexnet_3x3":          build_alexnet_3x3,
    "alexnet_2x2":          build_alexnet_2x2,
    "alexnet_stacked":      build_alexnet_stacked,
    "alexnet_mixed":        build_alexnet_mixed,
    "alexnet_small_kernel": build_alexnet_smallkernel,
}

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 9. Final comparison table

In [ ]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

In [14]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:22s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")


RANKING BY TOP-1 ACCURACY (all precisions)
 1. alexnet_fp32_INT8      [INT8] | top1= 22.08% | top5= 47.30% | params= 57.82M | size= 55.33MB
 2. AlexNet_FP32           [FP32] | top1= 19.92% | top5= 44.37% | params= 57.82M | size=661.75MB
 3. alexnet_small_kernel_INT8 [INT8] | top1= 10.77% | top5= 28.73% | params=  1.60M | size=  1.56MB
 4. AlexNetSmall_FP32      [FP32] | top1=  8.62% | top5= 25.49% | params=  1.60M | size= 18.35MB
 5. alexnet_3x3_INT8       [INT8] | top1=  7.52% | top5= 22.41% | params= 57.61M | size= 55.12MB
 6. AlexNet3x3_FP32        [FP32] | top1=  6.19% | top5= 19.25% | params= 57.61M | size=659.26MB


## 10. Persist experiment summary

In [15]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-phase2`** — one run per architecture, FP32 training
- **`qat-phase2`** — one run per architecture, QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```